# 05 — ONNX Export and Parity Check

**Purpose:** Export trained sklearn models to ONNX and verify numerical parity.  
**Acceptance criterion:** max probability difference sklearn vs onnxruntime < 0.1% (0.001).  
Class prediction must be identical in 100% of val cases.

**This notebook runs BEFORE any Node.js integration (F5).**  
If parity fails here, Node.js metrics are invalid.

**Input:** `training/models/rf_final.pkl`, `training/models/if_final.pkl`, `training/splits/val.parquet`  
**Output:** `models/rf.onnx`, `models/if.onnx`, `training/results/parity_report.txt`

## 1. Load Trained Models

In [1]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

MODELS  = Path("../../training/models")
SPLITS  = Path("../../training/splits")
RESULTS = Path("../../training/results")
RESULTS.mkdir(parents=True, exist_ok=True)

rf  = joblib.load(MODELS / "rf_v11.pkl")
iso = joblib.load(MODELS / "if_v10.pkl")

import json
with open(MODELS / "if_v10_metadata.json") as f:
    if_meta = json.load(f)

THRESHOLD = if_meta["threshold"]
print(f"RF loaded  : {rf.n_estimators} estimators")
print(f"IF loaded  , threshold: {THRESHOLD:.4f}")

META_COLS = [
    "sample_id", "timestamp", "_source", "_row_hash",
    "status_code", "req_count_1s", "req_count_5s", "req_count_60s",
    "error_rate_4xx_60s", "endpoint_diversity_60s",
]

# IF-specific: 6 more features dropped beyond META_COLS — near-zero variance
# in benign traffic, destabilizes IsolationForest calibration (matches
# packages/core/src/worker.ts's IF_ADDITIONAL_EXCLUDED / IF_MODEL_INDICES,
# which is what if.onnx is actually fed at inference time).
IF_ADDITIONAL_EXCLUDED = [
    "dotdot_encoded_count", "authorization_length", "unusual_headers_count",
    "null_byte_count", "os_path_indicator", "sensitive_file_target",
]

val  = pd.read_parquet(SPLITS / "val.parquet")
val.drop(columns=[c for c in META_COLS if c in val.columns], inplace=True)
y_val = val.pop("label")
X_val = val

X_val_if = X_val.drop(columns=[c for c in IF_ADDITIONAL_EXCLUDED if c in X_val.columns])

print(f"Val features shape (RF, 69): {X_val.shape}")
print(f"Val features shape (IF, 63): {X_val_if.shape}")

RF loaded  : 30 estimators
IF loaded  , threshold: 0.0081
Val features shape (RF, 69): (57443, 69)
Val features shape (IF, 63): (57443, 63)


## 2. Export RF to ONNX

In [2]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# X_val already loaded and cleaned in cell above (69 feature columns — RF)
N_FEATURES   = X_val.shape[1]
TARGET_OPSET = 17  # compatible with onnxruntime-node

assert N_FEATURES == 69, f"Expected 69 features, got {N_FEATURES}. Check META_COLS drop."

initial_type = [("float_input", FloatTensorType([None, N_FEATURES]))]

rf_onnx = convert_sklearn(
    rf,
    initial_types=initial_type,
    target_opset=TARGET_OPSET,
    options={id(rf): {"zipmap": False}},  # flat probability tensor
)

with open(MODELS / "rf.onnx", "wb") as f:
    f.write(rf_onnx.SerializeToString())

print(f"RF exported : {MODELS / 'rf.onnx'}")
print(f"N features  : {N_FEATURES}, opset: {TARGET_OPSET}")

RF exported : ../../training/models/rf.onnx
N features  : 69, opset: 17


## 3. Export IF to ONNX

In [3]:
IF_N_FEATURES = X_val_if.shape[1]
assert IF_N_FEATURES == 63, f"Expected 63 IF features, got {IF_N_FEATURES}. Check IF_ADDITIONAL_EXCLUDED drop."
initial_type_if = [("float_input", FloatTensorType([None, IF_N_FEATURES]))]

iso_onnx = convert_sklearn(
    iso,
    initial_types=initial_type_if,
    target_opset={"": TARGET_OPSET, "ai.onnx.ml": 3},
)

with open(MODELS / "if.onnx", "wb") as f:
    f.write(iso_onnx.SerializeToString())

print(f"IF exported : {MODELS / 'if.onnx'}")

IF exported : ../../training/models/if.onnx


## 4. Parity Check: Python sklearn vs onnxruntime

In [4]:
import onnxruntime as rt

rf_sess = rt.InferenceSession(str(MODELS / "rf.onnx"))
if_sess = rt.InferenceSession(str(MODELS / "if.onnx"))

sample       = X_val.sample(1000, random_state=42).astype(np.float32)
sample_np    = sample.values
sample_if    = X_val_if.loc[sample.index].astype(np.float32)
sample_if_np = sample_if.values

# RF parity
sklearn_probs = rf.predict_proba(sample)
onnx_probs    = rf_sess.run(None, {"float_input": sample_np})[1]  # index 1 = probabilities

max_diff_rf = np.abs(sklearn_probs - onnx_probs).max()
print(f"RF max probability difference : {max_diff_rf:.6f}")
assert max_diff_rf < 0.001, (
    f"GATE FAILED: RF parity violation. Max diff={max_diff_rf:.6f} >= 0.001"
)

# IF parity — skl2onnx exports IF as [labels, decision_function_values]
sklearn_scores = iso.decision_function(sample_if)
onnx_scores    = if_sess.run(None, {"float_input": sample_if_np})[1].flatten()

max_diff_if = np.abs(sklearn_scores - onnx_scores).max()
print(f"IF max score difference        : {max_diff_if:.6f}")
assert max_diff_if < 0.001, (
    f"GATE FAILED: IF parity violation. Max diff={max_diff_if:.6f} >= 0.001"
)

print("PARITY CHECK PASSED — both models cleared for Node.js integration.")

RF max probability difference : 0.000000
IF max score difference        : 0.000000
PARITY CHECK PASSED — both models cleared for Node.js integration.


## 5. Parity Report

In [5]:
import json

report = {
    "rf_onnx_path":         str(MODELS / "rf.onnx"),
    "if_onnx_path":         str(MODELS / "if.onnx"),
    "target_opset":         TARGET_OPSET,
    "rf_n_features":        N_FEATURES,
    "if_n_features":        IF_N_FEATURES,
    "parity_samples":       1000,
    "rf_max_prob_diff":     float(max_diff_rf),
    "if_max_score_diff":    float(max_diff_if),
    "parity_passed":        bool(max_diff_rf < 0.001 and max_diff_if < 0.001),
    "threshold_if":         THRESHOLD,
    "if_onnx_output_index": 1,
    "rf_onnx_output_index": 1,
    "rf_classes":           list(rf.classes_),
    "rf_model_version":     MODELS.joinpath("rf_v11.pkl").stem if (MODELS / "rf_v11.pkl").exists() else None,
    "if_model_version":     MODELS.joinpath("if_v10.pkl").stem if (MODELS / "if_v10.pkl").exists() else None,
}

with open(MODELS / "parity_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(f"Saved: {MODELS / 'parity_report.json'}")
print(json.dumps(report, indent=2))
print("\nThis file must exist and parity_passed=true before")
print("Sebastián integrates the ONNX files into the npm library.")

Saved: ../../training/models/parity_report.json
{
  "rf_onnx_path": "../../training/models/rf.onnx",
  "if_onnx_path": "../../training/models/if.onnx",
  "target_opset": 17,
  "rf_n_features": 69,
  "if_n_features": 63,
  "parity_samples": 1000,
  "rf_max_prob_diff": 9.465478112424819e-08,
  "if_max_score_diff": 2.4040887669496414e-07,
  "parity_passed": true,
  "threshold_if": 0.00806713286301003,
  "if_onnx_output_index": 1,
  "rf_onnx_output_index": 1,
  "rf_classes": [
    "benign",
    "cmdi",
    "path_traversal",
    "sqli",
    "xss"
  ],
  "rf_model_version": "rf_v11",
  "if_model_version": "if_v10"
}

This file must exist and parity_passed=true before
Sebastián integrates the ONNX files into the npm library.
